# Build a Tiny GPT and a Tiny AI Agent

**For first-year students.** No prior deep learning needed. Read the text, run each cell, and look at the output.

**What we will build**

| Step | Idea | What you will see |
|---|---|---|
| 1 | Words → numbers | Our tiny vocabulary |
| 2 | Attention | A word "looking at" other words, using simple hand-made numbers |
| 3 | Self-attention vs cross-attention | Looking inside the same sentence vs at another sentence |
| 4 | Embeddings | Each word becomes a list of numbers the model can learn |
| 5 | One attention head (Query, Key, Value) | A learnable attention layer |
| 6 | Masked self-attention | Why GPT cannot look at future words |
| 7 | Multi-head attention | Several heads looking in different ways |
| 8 | Feed-forward + Block | One complete GPT building block |
| 9 | Tiny GPT | Train it and make it write sentences |
| 10 | Tiny Agent | The GPT decides which tool to use; Python runs the tool |

**How to run:** open in Google Colab → *Runtime → Run all*. Everything runs on a normal CPU in about 2 minutes. No internet needed.

In [ ]:
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)   # so everyone gets the same results
random.seed(0)

def draw(weights, row_words, col_words, title):
    """Draws an attention table as a coloured picture. Brighter = more attention."""
    plt.figure(figsize=(len(col_words) * 0.7 + 2, len(row_words) * 0.5 + 1.5))
    plt.imshow(weights.detach(), cmap="viridis")
    plt.xticks(range(len(col_words)), col_words, rotation=45)
    plt.yticks(range(len(row_words)), row_words)
    plt.xlabel("looked at")
    plt.ylabel("looking")
    plt.title(title)
    plt.colorbar()
    plt.show()

## Step 1: Words → Numbers

Computers only understand numbers. So we:
1. Split text into words (**tokens**).
2. Give every different word an ID number. The list of all words is the **vocabulary**.

Below is our small training text. These are simple sentences we wrote ourselves.

In [ ]:
text = """
the cat sat on the mat .
the dog sat on the rug .
the cat ate the fish .
the dog ate the bone .
the cat is happy .
the dog is happy .
the bird sat on the tree .
the bird ate the seed .
the bird is small .
the cat is small .
the dog is big .
the cat ran to the dog .
the dog ran to the park .
the bird flew to the tree .
the cat slept on the mat .
the dog slept on the rug .
the bird sang a song .
the cat saw the bird .
the dog saw the cat .
the bird saw the fish .
"""

words = text.split()
vocab = sorted(set(words))

word_to_id = {w: i for i, w in enumerate(vocab)}
id_to_word = {i: w for i, w in enumerate(vocab)}

def encode(word_list):
    return [word_to_id[w] for w in word_list]

def decode(id_list):
    return " ".join(id_to_word[i] for i in id_list)

data = torch.tensor(encode(words))

print("Number of words in text:", len(words))
print("Vocabulary size:", len(vocab))
print("Vocabulary:", vocab)
print()
print("'the cat sat' as numbers:", encode(["the", "cat", "sat"]))
print("[5, 3] back to words:", decode([5, 3]))

## Step 2: Attention — the main idea

**Attention = a word looks at the other words and decides which ones are important to it.**

Example: in *"the **cat** sat because **it** was tired"*, the word **"it"** should look at **"cat"**.

Let's describe a few words with just **2 numbers** that we make up by hand:
- number 1 = "how much is this an animal?"
- number 2 = "how much is this an action?"

**How attention works:**
1. **Score:** compare two words by multiplying their numbers and adding (called a *dot product*). Similar words → big score.
2. **Softmax:** turn the scores into percentages that add up to 100%.
3. **Mix:** the new meaning of a word = a mix of the words it paid attention to, using those percentages.

In [ ]:
cat = [1.0, 0.0]    # an animal, not an action
sat = [0.0, 1.0]    # an action
it  = [0.9, 0.1]    # "it" is mostly like an animal

x = torch.tensor([cat, sat, it])
names = ["cat", "sat", "it"]

# 1. Score: compare every word with every word
scores = x @ x.T
print("Scores:")
print(scores)

# 2. Softmax: turn each row into percentages
weights = F.softmax(scores, dim=1)
print("\nAttention (each row adds up to 1):")
print(weights.round(decimals=2))

# 3. Mix: new meaning = weighted mix of the words
new_x = weights @ x
print("\nNew meaning of 'it':", new_x[2].round(decimals=2))

draw(weights, names, names, "Attention with hand-made numbers")
print("Look at the 'it' row: it pays the most attention to 'cat'.")

## Step 3: Self-attention vs cross-attention

The maths is the same. The only difference is **who looks at whom**:

| Type | Who is looking | Who is being looked at | Used in |
|---|---|---|---|
| **Self-attention** | words of a sentence | words of the **same** sentence | GPT, BERT |
| **Cross-attention** | words of sentence A | words of a **different** sentence B | Translation models (output looks at input) |

Step 2 was **self-attention** (cat, sat, it all looked at each other).

Now **cross-attention**: imagine translating to Hindi. The Hindi word **"बिल्ली"** (billi, cat) looks at the English words to find which one it matches.

In [ ]:
english = torch.tensor([cat, sat])           # the sentence being looked at
english_names = ["cat", "sat"]

billi = [1.0, 0.0]     # "बिल्ली" (cat) is an animal word
baitha = [0.0, 1.0]    # "बैठा" (sat) is an action word
hindi = torch.tensor([billi, baitha])        # the words that are looking
hindi_names = ["billi (cat)", "baitha (sat)"]

scores = hindi @ english.T                   # Hindi words compared with English words
weights = F.softmax(scores, dim=1)
draw(weights, hindi_names, english_names, "Cross-attention: Hindi words looking at English words")

## Step 4: Embeddings — let the model learn the numbers

In Step 2 **we** chose the numbers by hand. A real model **learns** them.

- **Token embedding:** a table with one row of numbers for each word. (`nn.Embedding`)
- **Position embedding:** a table with one row for each position (1st word, 2nd word, ...). Attention looks at all words at once, so without this the model would not know the word order. "dog bites man" and "man bites dog" would look the same!

Final input for each word = **word numbers + position numbers**.

At the start the numbers are random. Training will change them.

In [ ]:
emb_size = 8                    # each word becomes 8 numbers (real GPTs use 768 or more)

token_embedding = nn.Embedding(len(vocab), emb_size)
position_embedding = nn.Embedding(10, emb_size)    # up to 10 positions

sentence = ["the", "cat", "sat", "on", "the", "mat"]
ids = torch.tensor(encode(sentence))
positions = torch.arange(len(sentence))            # 0, 1, 2, 3, 4, 5

x = token_embedding(ids) + position_embedding(positions)

print("Sentence:", sentence)
print("Word IDs:", ids.tolist())
print("Positions:", positions.tolist())
print("Shape of x:", tuple(x.shape), "→ 6 words, 8 numbers each")
print("\nNumbers for 'cat':", x[1].detach().round(decimals=2))

## Step 5: One attention head — Query, Key, Value

In Step 2 we compared words directly. Real attention gives every word **three roles**, each made by a small learnable layer (`nn.Linear`):

| Role | Question | Library example |
|---|---|---|
| **Query** | What am I looking for? | What you type into the library search box |
| **Key** | What do I have? | The title on each book |
| **Value** | What do I give if chosen? | What is written inside the book |

Steps:
1. score = Query · Key (does what I want match what you have?)
2. Divide by √(size), so the numbers don't get too big.
3. Softmax → percentages.
4. Output = percentages × Values.

Below, `masked=False` means **every word can see every word**.

In [ ]:
class Head(nn.Module):
    """One attention head."""

    def __init__(self, emb_size, head_size, masked=True):
        super().__init__()
        self.query = nn.Linear(emb_size, head_size, bias=False)
        self.key = nn.Linear(emb_size, head_size, bias=False)
        self.value = nn.Linear(emb_size, head_size, bias=False)
        self.masked = masked

    def forward(self, x):
        q = self.query(x)                     # what each word is looking for
        k = self.key(x)                       # what each word has
        v = self.value(x)                     # what each word gives

        scores = q @ k.transpose(-2, -1)      # compare every query with every key
        scores = scores / (k.shape[-1] ** 0.5)

        if self.masked:                       # hide future words (Step 6)
            n = x.shape[-2]
            mask = torch.tril(torch.ones(n, n))
            scores = scores.masked_fill(mask == 0, float("-inf"))

        weights = F.softmax(scores, dim=-1)   # percentages
        self.weights = weights                # remember them so we can draw them
        return weights @ v                    # mix the values


head = Head(emb_size=8, head_size=8, masked=False)
out = head(x)
print("Input shape :", tuple(x.shape))
print("Output shape:", tuple(out.shape), "→ same size, but each word now carries information from other words")
draw(head.weights, sentence, sentence, "Self-attention (random, not trained yet)")

## Step 6: Masked self-attention — no peeking!

GPT writes **one word at a time**. When it is writing word 3, words 4, 5, 6 don't exist yet.

During training we show it the full sentence, so we must **hide future words**, or it would just copy the answer. This is called a **mask**.

`torch.tril` makes a triangle of 1s: each word can see **itself and the words before it**, never the words after.

In [ ]:
print("The mask (1 = allowed to see, 0 = hidden):")
print(torch.tril(torch.ones(6, 6)).int())

masked_head = Head(emb_size=8, head_size=8, masked=True)
masked_head(x)
draw(masked_head.weights, sentence, sentence, "MASKED self-attention: the top-right is always 0")
print("'the' (first word) can only see itself. 'mat' (last word) can see everything before it.")

## Step 7: Multi-head attention

One head can only focus on **one kind of relationship**. But words have many: *who did it? what action? where?*

**Multi-head attention = several heads working side by side.** Each head has its own Query, Key, Value, so each one learns to look for something different. At the end we join their outputs and mix them with one more layer.

Think of a group project: 4 students each check a different thing (grammar, facts, spelling, style), then the team leader combines their reports.

To keep cost the same, each head gets a smaller piece: with 8 numbers and 4 heads, each head uses 2 numbers.

In [ ]:
class MultiHead(nn.Module):
    """Several attention heads side by side."""

    def __init__(self, emb_size, num_heads):
        super().__init__()
        head_size = emb_size // num_heads
        self.heads = nn.ModuleList([Head(emb_size, head_size) for _ in range(num_heads)])
        self.mix = nn.Linear(emb_size, emb_size)

    def forward(self, x):
        outputs = [h(x) for h in self.heads]     # every head does its own attention
        joined = torch.cat(outputs, dim=-1)      # put the results side by side
        return self.mix(joined)                  # combine them


multi = MultiHead(emb_size=8, num_heads=4)
out = multi(x)
print("Output shape:", tuple(out.shape))

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for i, h in enumerate(multi.heads):
    axes[i].imshow(h.weights.detach(), cmap="viridis")
    axes[i].set_title(f"Head {i + 1}")
    axes[i].set_xticks(range(6)); axes[i].set_xticklabels(sentence, rotation=45)
    axes[i].set_yticks(range(6)); axes[i].set_yticklabels(sentence)
plt.suptitle("4 heads, 4 different attention patterns")
plt.tight_layout(); plt.show()

## Step 8: Feed-forward and the complete Block

**Feed-forward:** after attention *collects* information from other words, each word *thinks about it* on its own using a small neural network: make the numbers bigger (×4) → ReLU → shrink back.

> Attention = a team meeting where everyone shares. Feed-forward = everyone goes back to their desk to think.

**Block = Attention + Feed-forward**, plus two helpers:
- **Residual (`x + ...`):** keep the old information and *add* the new. Nothing gets lost.
- **LayerNorm:** keeps the numbers in a nice range so training stays stable.

GPT is just **many blocks stacked** on top of each other.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, emb_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_size, 4 * emb_size),   # expand
            nn.ReLU(),                           # keep positives, zero the negatives
            nn.Linear(4 * emb_size, emb_size),   # shrink back
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """One GPT block: attention, then feed-forward."""

    def __init__(self, emb_size, num_heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(emb_size)
        self.attention = MultiHead(emb_size, num_heads)
        self.norm2 = nn.LayerNorm(emb_size)
        self.feed_forward = FeedForward(emb_size)

    def forward(self, x):
        x = x + self.attention(self.norm1(x))      # words share information
        x = x + self.feed_forward(self.norm2(x))   # each word thinks
        return x


block = Block(emb_size=8, num_heads=4)
print("Block input :", tuple(x.shape))
print("Block output:", tuple(block(x).shape), "→ same shape, so we can stack as many blocks as we like")

## Step 9: The Tiny GPT

Now we put everything together:

```
word IDs → token embedding + position embedding → Block → Block → LayerNorm → Linear → a score for every word in the vocabulary
```

The last layer gives a score for **every word in the vocabulary**: "how likely is this word to come next?"

**GPT's only job: predict the next word.**

| Input | Correct next word |
|---|---|
| the | cat |
| the cat | sat |
| the cat sat | on |

In [ ]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size, emb_size=32, num_heads=4, num_blocks=2, max_len=16):
        super().__init__()
        self.max_len = max_len
        self.token_embedding = nn.Embedding(vocab_size, emb_size)
        self.position_embedding = nn.Embedding(max_len, emb_size)
        self.blocks = nn.Sequential(*[Block(emb_size, num_heads) for _ in range(num_blocks)])
        self.norm = nn.LayerNorm(emb_size)
        self.to_vocab = nn.Linear(emb_size, vocab_size)

    def forward(self, ids):
        positions = torch.arange(ids.shape[1])
        x = self.token_embedding(ids) + self.position_embedding(positions)
        x = self.blocks(x)
        x = self.norm(x)
        return self.to_vocab(x)        # scores for the next word at every position

### Training

We show the model many pieces of our text. For each word, the correct answer is simply **the word after it** (`y` is `x` moved one step).

1. Model guesses the next words.
2. **Loss** measures how wrong the guesses are.
3. `loss.backward()` works out how to change every number in the model.
4. The optimizer changes the numbers a tiny bit.
5. Repeat many times. The loss should go **down**.

In [ ]:
def train(model, data, steps=1500, batch_size=32):
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
    n = model.max_len
    losses = []
    for step in range(steps):
        starts = torch.randint(0, len(data) - n - 1, (batch_size,))
        x = torch.stack([data[s : s + n] for s in starts])           # input words
        y = torch.stack([data[s + 1 : s + n + 1] for s in starts])   # the next words (answers)

        scores = model(x)
        loss = F.cross_entropy(scores.view(-1, scores.shape[-1]), y.view(-1))

        optimizer.zero_grad()
        loss.backward()        # find how to improve
        optimizer.step()       # improve a little

        losses.append(loss.item())
        if step % 300 == 0:
            print(f"step {step:5d}   loss {loss.item():.3f}")

    plt.figure(figsize=(6, 3))
    plt.plot(losses)
    plt.xlabel("training step"); plt.ylabel("loss"); plt.title("Loss going down = model learning")
    plt.show()


gpt = TinyGPT(vocab_size=len(vocab))
print("Number of learnable numbers (parameters):", sum(p.numel() for p in gpt.parameters()))
print("(For comparison, GPT-3 has 175,000,000,000.)\n")
train(gpt, data)

### Let it write!

**Generating text = predict one word, add it, repeat.** This is exactly how ChatGPT and Claude write.

- `greedy=True` → always pick the most likely word.
- `greedy=False` → pick randomly, but likely words more often. This gives more variety.

In [ ]:
def generate(model, start_ids, max_new_words=10, stop_id=None, greedy=False):
    model.eval()
    ids = list(start_ids)
    for _ in range(max_new_words):
        x = torch.tensor([ids[-model.max_len:]])          # the words so far
        with torch.no_grad():
            scores = model(x)[0, -1]                      # scores for the NEXT word
        probs = F.softmax(scores, dim=-1)                 # scores → probabilities
        if greedy:
            next_id = torch.argmax(probs).item()
        else:
            next_id = torch.multinomial(probs, 1).item()
        ids.append(next_id)
        if next_id == stop_id:
            break
    return ids


for start in ["the cat", "the dog", "the bird"]:
    ids = generate(gpt, encode(start.split()), max_new_words=12)
    print(decode(ids))

### What does the model think comes next?

In [ ]:
prompt = "the cat sat on the"
with torch.no_grad():
    scores = gpt(torch.tensor([encode(prompt.split())]))[0, -1]
probs = F.softmax(scores, dim=-1)
top = torch.topk(probs, 5)

print(f"After '{prompt}', the model's top guesses:")
for p, i in zip(top.values, top.indices):
    print(f"  {id_to_word[i.item()]:>6}  {p.item():.0%}")

## Step 10: From GPT to an AI Agent

A GPT can only **write text**. It cannot do maths reliably, check the weather, or search the internet.

**An AI agent = GPT (the brain) + tools (the hands) + a loop.**

```
User question
    ↓
GPT decides:  answer directly?  or  use a tool?   ← the "brain"
    ↓
Python runs the tool (calculator, weather, ...)   ← the "hands"
    ↓
Final answer to the user
```

This is how ChatGPT runs code, how Claude searches the web, and how coding assistants edit files: the model **writes a tool request** and normal code **carries it out**.

**Our plan:** train a fresh Tiny GPT to read a question and write a short *decision* like:

| Question | GPT writes |
|---|---|
| what is 12 plus 7 ? | `CALL add 12 7` |
| 4 times 9 | `CALL multiply 4 9` |
| weather in chennai ? | `CALL weather chennai` |
| hello | `SAY hello ! how can i help you ?` |

Notice the model must **copy the right numbers and city** from the question. That is attention at work!

### 10a. The tools (normal Python functions)

In [ ]:
def add(a, b):
    return int(a) + int(b)

def multiply(a, b):
    return int(a) * int(b)

# Made-up weather values, just for this demo (not real data)
fake_weather = {
    "chennai": "33°C, sunny",
    "delhi": "28°C, hazy",
    "mumbai": "30°C, rainy",
    "kolkata": "31°C, humid",
    "bengaluru": "24°C, cloudy",
    "hyderabad": "29°C, clear",
}

def weather(city):
    return fake_weather[city]

TOOLS = {"add": add, "multiply": multiply, "weather": weather}

print(add("12", "7"), "|", multiply("4", "9"), "|", weather("chennai"))

### 10b. Make training examples

We automatically create thousands of example conversations. Each one looks like:

`<q> what is 12 plus 7 ? <a> CALL add 12 7 <end>`

- `<q>` = question starts, `<a>` = answer starts, `<end>` = stop writing.

In [ ]:
numbers = [str(n) for n in range(21)]          # 0 to 20
cities = list(fake_weather.keys())

def make_example():
    kind = random.choice(["add", "multiply", "weather", "hello"])
    a, b = random.choice(numbers), random.choice(numbers)
    city = random.choice(cities)

    if kind == "add":
        question = random.choice([f"what is {a} plus {b} ?", f"add {a} and {b}", f"{a} plus {b}"])
        answer = f"CALL add {a} {b}"
    elif kind == "multiply":
        question = random.choice([f"what is {a} times {b} ?", f"multiply {a} and {b}", f"{a} times {b}"])
        answer = f"CALL multiply {a} {b}"
    elif kind == "weather":
        question = random.choice([f"weather in {city} ?", f"how is the weather in {city} ?", f"is it hot in {city} ?"])
        answer = f"CALL weather {city}"
    else:
        question = random.choice(["hello", "hi", "good morning"])
        answer = "SAY hello ! how can i help you ?"

    return f"<q> {question} <a> {answer} <end>"


examples = [make_example() for _ in range(3000)]
for e in examples[:5]:
    print(e)

agent_words = " ".join(examples).split()
agent_vocab = sorted(set(agent_words))
a_word_to_id = {w: i for i, w in enumerate(agent_vocab)}
a_id_to_word = {i: w for i, w in enumerate(agent_vocab)}
agent_data = torch.tensor([a_word_to_id[w] for w in agent_words])

print("\nAgent vocabulary size:", len(agent_vocab))

### 10c. Train the agent's brain

Same `TinyGPT` class, same `train` function. Only the training text is different. This is a key lesson: **the architecture stays the same; the data decides what the model learns.**

In [ ]:
torch.manual_seed(0)
brain = TinyGPT(vocab_size=len(agent_vocab), emb_size=64, num_heads=4, num_blocks=2, max_len=24)
train(brain, agent_data, steps=2000)

### 10d. The agent loop

1. Turn the question into IDs.
2. The GPT writes its decision, word by word, until `<end>`.
3. If it says `CALL`, Python runs that tool.
4. If it says `SAY`, we just reply with its words.

*Note:* real agents also send the tool's result **back to the model** so it can write a natural answer, and may loop several times. We keep it to one step so the code stays short.

In [ ]:
def agent(question):
    print("User  :", question)

    words = question.lower().split()
    unknown = [w for w in words if w not in a_word_to_id]
    if unknown:
        print("Agent : sorry, I don't know these words:", unknown, "\n")
        return

    prompt = ["<q>"] + words + ["<a>"]
    prompt_ids = [a_word_to_id[w] for w in prompt]

    # 1. The brain (GPT) decides what to do
    out_ids = generate(brain, prompt_ids, max_new_words=10,
                       stop_id=a_word_to_id["<end>"], greedy=True)
    decision = [a_id_to_word[i] for i in out_ids[len(prompt_ids):] if a_id_to_word[i] != "<end>"]
    print("Brain :", " ".join(decision))

    # 2. Act on the decision
    if decision[0] == "CALL":
        tool_name, args = decision[1], decision[2:]
        result = TOOLS[tool_name](*args)              # the hands: run the real Python function
        print(f"Tool  : {tool_name}({', '.join(args)}) → {result}")
        print("Agent : The answer is", result, "\n")
    else:
        print("Agent :", " ".join(decision[1:]), "\n")


agent("what is 12 plus 7 ?")
agent("multiply 6 and 9")
agent("how is the weather in mumbai ?")
agent("hello")
agent("17 times 3")
agent("what is the capital of france ?")

### 10e. How well did it learn?

We test on 300 brand-new random questions and count how many decisions are exactly right.

In [ ]:
correct = 0
tests = 300
for _ in range(tests):
    example = make_example().split()
    q_end = example.index("<a>") + 1
    prompt_ids = [a_word_to_id[w] for w in example[:q_end]]
    out_ids = generate(brain, prompt_ids, max_new_words=10, stop_id=a_word_to_id["<end>"], greedy=True)
    predicted = [a_id_to_word[i] for i in out_ids[q_end:]]
    if predicted == example[q_end:]:
        correct += 1

print(f"Correct decisions: {correct} / {tests}  ({correct / tests:.0%})")

### 10f. Look inside: attention copying the numbers

When the brain writes the numbers after `CALL add`, it has to **find them in the question**. The picture shows where each word **looked** (brighter = more attention).

**How to read it:** each row is the word the model is standing on while it predicts the **next** word. Look at the bottom rows:
- Row **CALL** looks at **plus** → so the next word it writes is **add**.
- Row **add** looks at **14** in the question → so it writes **14**.
- Row **14** looks at **5** in the question → so it writes **5**.

Nobody wrote a rule saying "copy the numbers". The model **learned** to use attention this way during training.

In [ ]:
question = "what is 14 plus 5 ?"
prompt = ["<q>"] + question.split() + ["<a>"]
prompt_ids = [a_word_to_id[w] for w in prompt]
out_ids = generate(brain, prompt_ids, max_new_words=10, stop_id=a_word_to_id["<end>"], greedy=True)
all_words = [a_id_to_word[i] for i in out_ids]

with torch.no_grad():
    brain(torch.tensor([out_ids]))         # run once more so every head saves its attention

last_block = brain.blocks[-1]
avg = sum(h.weights[0] for h in last_block.attention.heads) / len(last_block.attention.heads)
draw(avg, all_words, all_words, "Brain's attention (last block, average of heads)")

## Summary

| Idea | One-line meaning | In our code |
|---|---|---|
| Token | A word turned into an ID number | `encode`, `decode` |
| Attention | A word looks at other words and mixes in what's important | Step 2 |
| Self-attention | Words look at words in the **same** sentence | `Head(masked=False)` |
| Cross-attention | Words look at a **different** sentence | Step 3 |
| Embedding | Learnable numbers for each word and position | `nn.Embedding` |
| Query / Key / Value | What I want / what I have / what I give | `Head` |
| Masked self-attention | Can't see future words | `Head(masked=True)`, `torch.tril` |
| Multi-head attention | Several heads looking for different things | `MultiHead` |
| Feed-forward | Each word "thinks" on its own | `FeedForward` |
| Block | Attention + feed-forward (+ residual + LayerNorm) | `Block` |
| GPT | Stack of blocks that predicts the next word | `TinyGPT` |
| Agent | GPT decides, tools act | `agent()` |

## Try it yourself

1. In Step 9, change `num_blocks=2` to `num_blocks=1`. Does the writing get worse?
2. Add your own sentences to `text` (e.g. about a cow or a fish) and retrain. Does the GPT use the new words?
3. In Step 6, set `masked=False` inside `Head` (default value), retrain the Tiny GPT and generate again. What goes wrong? Why?
4. Add a new tool, `subtract`. You need to: write the function, add it to `TOOLS`, add `minus` questions in `make_example`, and retrain.
5. Ask the agent `what is 25 plus 3 ?`. Why does it fail? (Hint: look at the `numbers` list.)